testing is there need of  PCA

In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.multioutput import RegressorChain
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score

In [2]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")


In [3]:
df = pd.read_csv('preprocessed.csv')
df.set_index('Date', inplace=True)
cat = df.select_dtypes(include=['object']).columns.tolist()

x = df.drop(['Units Sold', 'Demand'], axis=1)
y = df[['Units Sold', 'Demand']]

x_train, x_test, y_train, y_test = train_test_split(
    x, y,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

C:\Users\kumar\AppData\Local\Temp\ipykernel_10744\3298373648.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat = df.select_dtypes(include=['object']).columns.tolist()


In [4]:
ct = ColumnTransformer(
    transformers=[
        ('encoder', OneHotEncoder(), cat)
    ],
    remainder='passthrough'
)

x_train = ct.fit_transform(x_train)
x_test = ct.transform(x_test)

x_train[np.isinf(x_train)] = np.nan
x_test[np.isinf(x_test)] = np.nan

imputer = SimpleImputer(strategy='median')

x_train = imputer.fit_transform(x_train)
x_test = imputer.transform(x_test)


In [5]:
from sklearn.decomposition import PCA
pca = PCA(n_components=0.95)
x_train = pca.fit_transform(x_train)
x_test = pca.transform(x_test)

In [6]:
x_train.shape, x_test.shape

((60240, 9), (15060, 9))

In [7]:


xgb_model = RegressorChain(
    XGBRegressor(
        objective='reg:squarederror',
        random_state=42,
        reg_lambda=1.5,
        learning_rate=0.1,
        n_estimators=600,
        max_depth=5,
        min_child_weight=2,
        gamma=0.1,
        subsample=0.8,
        colsample_bytree=0.8
    )
)

# train
xgb_model.fit(x_train, y_train)

# train prediction
y_pred_train = xgb_model.predict(x_train)

mse = mean_squared_error(y_train, y_pred_train)
r2 = r2_score(y_train, y_pred_train)

print(f"Train RMSE: {np.sqrt(mse)}")
print(f"Train R2: {r2}")

# test prediction
y_pred_test = xgb_model.predict(x_test)

mse = mean_squared_error(y_test, y_pred_test)
r2 = r2_score(y_test, y_pred_test)

print(f"Test RMSE: {np.sqrt(mse)}")
print(f"Test R2: {r2}")

Train RMSE: 33.94329067052909
Train R2: 0.44705978416263575
Test RMSE: 38.72740346805404
Test R2: 0.26549821581248695
